In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
print(len(words))
words[:8]

32033


['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [4]:
block_size = 3
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        #print(context)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)
        

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [5]:
C = torch.randn((27, 2))

In [6]:
emb = C[X]
#print(emb)
print(emb[:,0,:][0])
#print(torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], dim=1))
print(torch.unbind(emb, 1))

tensor([0.2278, 0.2537])
(tensor([[ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [-0.4270,  0.1583],
        [-0.3993,  0.9461],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 0.5178, -1.5916],
        [ 0.2974,  1.0959],
        [-0.3412,  1.3251],
        [-0.6957, -0.2583],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 1.8365,  0.9935],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [-0.3412,  1.3251],
        [ 1.2935, -0.0189],
        [ 1.8365,  0.9935],
        [ 2.2474,  1.9632],
        [-0.4270,  0.1583],
        [ 0.2974,  1.0959],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [ 1.2935, -0.0189],
        [ 0.5178, -1.5916],
        [-0.0810,  0.2564],
        [-0.4498,  0.4370]]), tensor([[ 0.2278,  0.2537],
        [ 0.2278,  0.2537],
        [-0.4270,  0

In [7]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

We have to reshape the embeddings so that their shape matches W1, because we want to multiply them using matrix multiplication.
One way is to use unbind() and cat():

In [8]:
print(torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape)
# or use unbind
print(torch.cat(torch.unbind(emb, 1), 1).shape)

torch.Size([32, 6])
torch.Size([32, 6])


Another way is to use view():

In [10]:
emb.view(32, 6)[0]

tensor([0.2278, 0.2537, 0.2278, 0.2537, 0.2278, 0.2537])

view() is more efficient because it usually doesn't move or copy the actual data. Instead, it creates a new Tensor that shares the same underlying storage while interpreting the data with different dimensions.

In [ ]:
# More Respectable:

In [15]:
emb = C[X]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2

# Now you can use:

# counts = logits.exp()
# prob = counts / counts.sum(1, keepdim=True)
# loss = -prob[torch.arange(32), Y].log().mean()

# Or
loss = F.cross_entropy(logits, Y)
loss

tensor(29.6480)

F.cross_entropy(logits, Y) is better because it is numerically stable, more efficient, simpler and cleaner, and combines softmax + log + negative log-likelihood internally.

F.cross_entropy(logits, targets) converts logits into class probabilities using softmax, takes the probability of the correct class, applies the negative log, and averages the losses across the batch.

Logits → Softmax → Correct-class probability → Negative log → Mean loss

In [16]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

Using Mini-Batch and testing different learning rates to find the best

In [17]:
for p in parameters:
  p.requires_grad = True

In [18]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
lri = []
lossi = []

for i in range(10000):
     # minibatch construct
    ix = torch.randint(0, X.shape[0], (32,))
  
    # forward pass
    emb = C[X[ix]] # (32, 3, 10)
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y[ix])
    #print(loss.item())
  
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
  
    # update
    #lr = lrs[i]
    lr = 0.1
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    lri.append(lre[i])
    lossi.append(loss.item())

#print(loss.item())